# Validate the process surrogate

Run from `notebooks/` after generating data and training the model. Evaluation and gradients use only the held-out last 15% of the time series. All plots are exported to `results/validation_*.png`. The saved network is a **static** surrogate: residuals over time matter for control, and its gradients describe local *model* sensitivity, not causal or dynamic plant gain. One-at-a-time and contour sweeps can leave the observed joint operating envelope.

In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch import nn

DATA_PATH = Path('../data/synthetic_process_data.csv')
MODEL_DIR = Path('../models')
RESULTS_DIR = Path('../results')
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Generate data first: {DATA_PATH.resolve()}')
checkpoint = torch.load(MODEL_DIR / 'process_model.pt', map_location='cpu', weights_only=True)
# Pickle is unsafe for untrusted files; use only the scalers produced locally by notebook 02.
with (MODEL_DIR / 'input_scaler.pkl').open('rb') as file:
    input_scaler = pickle.load(file)
with (MODEL_DIR / 'output_scaler.pkl').open('rb') as file:
    output_scaler = pickle.load(file)
input_columns = checkpoint['input_columns']
output_columns = checkpoint['output_columns']
hidden_sizes = checkpoint['hidden_sizes']
if (len(input_columns), len(output_columns), tuple(hidden_sizes)) != (10, 2, (32, 32, 16)):
    raise ValueError('Unexpected checkpoint architecture or column count')
if list(input_scaler.feature_names_in_) != input_columns or list(output_scaler.feature_names_in_) != output_columns:
    raise ValueError('Checkpoint and scaler column orders disagree')

layers = []
for in_features, out_features in zip((len(input_columns), *hidden_sizes), (*hidden_sizes, len(output_columns))):
    layers.append(nn.Linear(in_features, out_features))
    if out_features != len(output_columns):
        layers.append(nn.SiLU())

class ProcessModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(*layers)

    def forward(self, inputs):
        return self.network(inputs)

model = ProcessModel()
model.load_state_dict(checkpoint['model_state_dict'], strict=True)
model.eval()
data = pd.read_csv(DATA_PATH)
missing = set(input_columns + output_columns) - set(data.columns)
if missing:
    raise ValueError(f'Missing data columns: {sorted(missing)}')
if len(data) != 8000 or not np.isfinite(data[input_columns + output_columns].to_numpy(dtype=float)).all():
    raise ValueError('Expected 8,000 finite sequential observations')
train = data.iloc[:int(0.70 * len(data))]
test = data.iloc[int(0.85 * len(data)):]
if not np.allclose(input_scaler.mean_, train[input_columns].mean().to_numpy(), rtol=1e-5):
    raise ValueError('Input scaler does not match the chronological training partition')
if not np.allclose(output_scaler.mean_, train[output_columns].mean().to_numpy(), rtol=1e-5):
    raise ValueError('Output scaler does not match the chronological training partition')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Loaded best training epoch {checkpoint["best_epoch"]}; held-out test rows: {len(test)}')

In [ ]:
def predict_physical(inputs):
    """Predict outputs in plant units from ordered physical input rows."""
    scaled = input_scaler.transform(pd.DataFrame(inputs, columns=input_columns)).astype(np.float32)
    with torch.no_grad():
        prediction = model(torch.from_numpy(scaled)).numpy()
    return output_scaler.inverse_transform(prediction)

actual = test[output_columns].to_numpy()
predicted = predict_physical(test[input_columns].to_numpy())
residuals = actual - predicted
metrics = pd.DataFrame([
    {'output': name, 'r2': r2_score(actual[:, i], predicted[:, i]),
     'mae': mean_absolute_error(actual[:, i], predicted[:, i]),
     'rmse': np.sqrt(mean_squared_error(actual[:, i], predicted[:, i]))}
    for i, name in enumerate(output_columns)
])
display(metrics)
metrics.to_csv(RESULTS_DIR / 'validation_metrics.csv', index=False)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes, output_columns)):
    limits = [min(actual[:, i].min(), predicted[:, i].min()),
              max(actual[:, i].max(), predicted[:, i].max())]
    ax.scatter(actual[:, i], predicted[:, i], s=8, alpha=0.4)
    ax.plot(limits, limits, 'k--', linewidth=1)
    ax.set(xlabel=f'Actual {name}', ylabel=f'Predicted {name}', title=name)
fig.suptitle('Held-out actual vs predicted')
fig.savefig(RESULTS_DIR / 'validation_parity.png', dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes, output_columns)):
    ax.hist(residuals[:, i], bins=40, density=True, alpha=0.8)
    ax.axvline(0, color='black', linestyle='--')
    ax.set(xlabel='Actual - predicted', ylabel='Density', title=name)
fig.suptitle('Held-out residual distributions')
fig.savefig(RESULTS_DIR / 'validation_residual_distributions.png', dpi=150)
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True, constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes, output_columns)):
    ax.plot(test.index, residuals[:, i], linewidth=0.7)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.set(ylabel=f'{name}\nactual - predicted')
axes[-1].set_xlabel('Sequential sample index')
fig.suptitle('Held-out residuals over time')
fig.savefig(RESULTS_DIR / 'validation_residuals_vs_time.png', dpi=150)
plt.show()

for i, name in enumerate(output_columns):
    fig, axes = plt.subplots(2, 5, figsize=(18, 7), constrained_layout=True)
    for ax, feature in zip(axes.flat, input_columns):
        ax.scatter(test[feature], residuals[:, i], s=7, alpha=0.35)
        ax.axhline(0, color='black', linestyle='--', linewidth=1)
        ax.set(xlabel=feature, ylabel='Actual - predicted')
    fig.suptitle(f'{name}: held-out residuals vs each input')
    fig.savefig(RESULTS_DIR / f'validation_residuals_vs_inputs_{name}.png', dpi=150)
    plt.show()


## Local automatic-differentiation sensitivities

For each test observation, compute the full 2 × 10 Jacobian. `dy/dx` uses physical output per physical input unit: `d(y_scaled)/d(x_scaled) × output_scaler.scale_ / input_scaler.scale_`. Rankings use the mean absolute **standardized** gradient, equivalent to a change in output training standard deviations per input training standard deviation; signed means expose direction and cancellation. Rankings are model-local, not a causal intervention study.

In [ ]:
scaled_test = input_scaler.transform(test[input_columns]).astype(np.float32)
features = torch.tensor(scaled_test, requires_grad=True)
scaled_outputs = model(features)
standardized_jacobian = np.stack([
    torch.autograd.grad(scaled_outputs[:, i].sum(), features,
                        retain_graph=i < len(output_columns) - 1)[0].detach().numpy()
    for i in range(len(output_columns))
], axis=1)
physical_jacobian = (standardized_jacobian * output_scaler.scale_[None, :, None]
                     / input_scaler.scale_[None, None, :])
if not np.isfinite(physical_jacobian).all():
    raise ValueError('Non-finite model sensitivities')

gradient_rows = []
ranking_rows = []
for i, name in enumerate(output_columns):
    for j, feature in enumerate(input_columns):
        gradient_rows.append(pd.DataFrame({
            'sample_index': test.index, 'output': name, 'input': feature,
            'dy_dx': physical_jacobian[:, i, j],
            'standardized_dy_dx': standardized_jacobian[:, i, j],
        }))
        ranking_rows.append({
            'output': name, 'input': feature,
            'mean_absolute_standardized_gradient': np.abs(standardized_jacobian[:, i, j]).mean(),
            'mean_signed_standardized_gradient': standardized_jacobian[:, i, j].mean(),
            'mean_absolute_dy_dx': np.abs(physical_jacobian[:, i, j]).mean(),
            'mean_signed_dy_dx': physical_jacobian[:, i, j].mean(),
        })
gradient_table = pd.concat(gradient_rows, ignore_index=True)
ranking = pd.DataFrame(ranking_rows).sort_values(
    ['output', 'mean_absolute_standardized_gradient'], ascending=[True, False])
ranking['rank'] = ranking.groupby('output').cumcount() + 1
display(ranking)
gradient_table.to_csv(RESULTS_DIR / 'validation_test_gradients.csv', index=False)
ranking.to_csv(RESULTS_DIR / 'validation_sensitivity_ranking.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for ax, name in zip(axes, output_columns):
    ordered = ranking[ranking['output'] == name].iloc[::-1]
    ax.barh(ordered['input'], ordered['mean_absolute_standardized_gradient'])
    ax.set(xlabel='Mean absolute standardized gradient', title=name)
fig.suptitle('Local sensitivity rankings on held-out observations')
fig.savefig(RESULTS_DIR / 'validation_sensitivity_rankings.png', dpi=150)
plt.show()


## Conditional response surfaces

Fix all other inputs to their training medians and sweep each input over its training 5th–95th percentile. Contours show interactions between the two most sensitive **manipulated** variables for each output, ranked using held-out gradients. These static counterfactuals may not represent feasible closed-loop trajectories or causal effects.

In [ ]:
baseline = train[input_columns].median().to_numpy(dtype=float)
ranges = train[input_columns].quantile([0.05, 0.95])
for output_index, name in enumerate(output_columns):
    fig, axes = plt.subplots(2, 5, figsize=(18, 7), constrained_layout=True)
    for feature_index, (ax, feature) in enumerate(zip(axes.flat, input_columns)):
        sweep = np.linspace(*ranges[feature].to_numpy(), 100)
        inputs = np.tile(baseline, (len(sweep), 1))
        inputs[:, feature_index] = sweep
        ax.plot(sweep, predict_physical(inputs)[:, output_index])
        ax.axvline(baseline[feature_index], color='gray', linestyle='--', linewidth=1)
        ax.set(xlabel=feature, ylabel=name)
    fig.suptitle(f'{name}: one-at-a-time response at training medians')
    fig.savefig(RESULTS_DIR / f'validation_oat_{name}.png', dpi=150)
    plt.show()


In [ ]:
controls = input_columns[5:]
for output_index, name in enumerate(output_columns):
    top = ranking[(ranking['output'] == name) & ranking['input'].isin(controls)]['input'].head(2).tolist()
    x_name, y_name = top
    x_grid, y_grid = np.meshgrid(np.linspace(*ranges[x_name].to_numpy(), 75),
                                 np.linspace(*ranges[y_name].to_numpy(), 75))
    inputs = np.tile(baseline, (x_grid.size, 1))
    inputs[:, input_columns.index(x_name)] = x_grid.ravel()
    inputs[:, input_columns.index(y_name)] = y_grid.ravel()
    surface = predict_physical(inputs)[:, output_index].reshape(x_grid.shape)
    fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
    contour = ax.contourf(x_grid, y_grid, surface, levels=20, cmap='viridis')
    ax.contour(x_grid, y_grid, surface, levels=10, colors='white', linewidths=0.5, alpha=0.7)
    ax.set(xlabel=x_name, ylabel=y_name, title=f'{name}: most sensitive control interaction')
    fig.colorbar(contour, ax=ax, label=name)
    fig.savefig(RESULTS_DIR / f'validation_contour_{name}.png', dpi=150)
    plt.show()
